In [2]:
# [FILE: notebooks/02_tien_xu_ly.ipynb]

# Thư viện xử lý dữ liệu
import pandas as pd

# Đọc dữ liệu gốc
df = pd.read_csv("../data/raw/houses_hcm_10000_raw.csv")

# Hiển thị số dòng và số cột
print("Kích thước dữ liệu:", df.shape)

Kích thước dữ liệu: (10000, 19)


In [3]:
# Tạo bản sao để xử lý
df_clean = df.copy()

# Hiển thị số dòng trước khi làm sạch
print("Số dòng trước khi làm sạch:", len(df_clean))

Số dòng trước khi làm sạch: 10000


In [6]:
# kiểm tra dữ liệu thiếu 
# Đếm số giá trị thiếu ở từng cột
missing = df_clean.isnull().sum()

# Chỉ hiển thị các cột có dữ liệu thiếu
print("CÁC CỘT CÓ DỮ LIỆU THIẾU:")
print(missing[missing > 0])

CÁC CỘT CÓ DỮ LIỆU THIẾU:
district_name         636
ward_name            1631
street_name          3306
project_name         7574
floor_count          7004
frontage_width       5409
house_depth          9931
road_width           8334
bedroom_count        2869
bathroom_count       3253
house_direction      8037
balcony_direction    8677
dtype: int64


In [8]:
# Tính tỷ lệ thiếu theo từng cột
missing_percent = (
    df_clean.isnull().mean() * 100
).sort_values(ascending=False)

# Chỉ hiển thị các cột có dữ liệu thiếu
print("TỶ LỆ DỮ LIỆU THIẾU:")
print(missing_percent[missing_percent > 0])

TỶ LỆ DỮ LIỆU THIẾU:
house_depth          99.319932
balcony_direction    86.778678
road_width           83.348335
house_direction      80.378038
project_name         75.747575
floor_count          70.047005
frontage_width       54.095410
street_name          33.063306
bathroom_count       32.533253
bedroom_count        28.692869
ward_name            16.311631
district_name         6.360636
dtype: float64


In [9]:
# xử lý dữ liệu thiếu 
# Lấy các cột dạng số
numeric_cols = df_clean.select_dtypes(include="number").columns

# Lấy các cột dạng chữ
categorical_cols = df_clean.select_dtypes(include="object").columns

# Điền giá trị thiếu của cột số bằng trung vị
for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Điền giá trị thiếu của cột chữ bằng "Không rõ"
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna("Không rõ")

print("Đã xử lý dữ liệu thiếu.")

Đã xử lý dữ liệu thiếu.


C:\Users\Tan Loi\AppData\Local\Temp\ipykernel_19760\508294826.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_clean.select_dtypes(include="object").columns


In [10]:
#  loại giá bất thường 
# Giá > 1000 tỷ nhưng diện tích < 500 m²
bat_thuong = (
    (df_clean["price"] > 1000e9) &
    (df_clean["area"] < 500)
)

# Đếm số dòng bất thường
print("Số dòng bất thường:", bat_thuong.sum())

# Hiển thị các dòng bất thường
display(
    df_clean.loc[bat_thuong, [
        "price",
        "area",
        "district_name",
        "property_type_name",
        "description"
    ]]
)

# Loại các dòng bất thường
df_clean = df_clean[~bat_thuong].copy()

# Kiểm tra số dòng còn lại
print("Số dòng sau khi làm sạch:", len(df_clean))

Số dòng bất thường: 0


,price,area,district_name,property_type_name,description


Số dòng sau khi làm sạch: 9999


In [11]:
# Kiểm tra lại dữ liệu thiếu sau khi xử lý
missing_after = df_clean.isnull().sum()

print("DỮ LIỆU THIẾU SAU KHI XỬ LÝ:")
print(missing_after[missing_after > 0])

DỮ LIỆU THIẾU SAU KHI XỬ LÝ:
Series([], dtype: int64)


In [12]:
# kiểm tra dữ liệu trước khi làm sạch 
# Kiểm tra kích thước dữ liệu
print("Kích thước sau khi làm sạch:", df_clean.shape)
# Kiểm tra thông tin các cột
df_clean.info()

Kích thước sau khi làm sạch: (9999, 19)
<class 'pandas.DataFrame'>
Index: 9999 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   name                9999 non-null   str    
 1   description         9999 non-null   str    
 2   property_type_name  9999 non-null   str    
 3   province_name       9999 non-null   str    
 4   district_name       9999 non-null   str    
 5   ward_name           9999 non-null   str    
 6   street_name         9999 non-null   str    
 7   project_name        9999 non-null   str    
 8   price               9999 non-null   float64
 9   area                9999 non-null   float64
 10  floor_count         9999 non-null   float64
 11  frontage_width      9999 non-null   float64
 12  house_depth         9999 non-null   float64
 13  road_width          9999 non-null   float64
 14  bedroom_count       9999 non-null   float64
 15  bathroom_count      9999 non-nu

In [13]:
# kiểm tra lại giá 
# Thống kê giá sau khi làm sạch
print("THỐNG KÊ GIÁ SAU KHI LÀM SẠCH:")
print(df_clean["price"].describe())

THỐNG KÊ GIÁ SAU KHI LÀM SẠCH:
count    9.999000e+03
mean     1.754667e+10
std      4.047976e+10
min      4.850000e+08
25%      5.099500e+09
50%      7.720000e+09
75%      1.400000e+10
max      1.250000e+12
Name: price, dtype: float64


In [15]:
# Lưu dữ liệu đã làm sạch
df_clean.to_csv(
    "../data/processed/houses_clean.csv",
    index=False
)

print("Đã lưu dữ liệu sạch.")
print("File: data/processed/houses_clean.csv")

Đã lưu dữ liệu sạch.
File: data/processed/houses_clean.csv


In [16]:
#  kiem tra file kết quả 
# Đọc lại file đã lưu để kiểm tra
df_check = pd.read_csv(
    "../data/processed/houses_clean.csv"
)

print("Kích thước file sau khi lưu:", df_check.shape)

# Xem 5 dòng đầu
display(df_check.head())

Kích thước file sau khi lưu: (9999, 19)


,name,description,property_type_name,province_name,district_name,ward_name,street_name,project_name,price,area,floor_count,frontage_width,house_depth,road_width,bedroom_count,bathroom_count,house_direction,balcony_direction,published_at
0,"Bán căn hộ dịch vụ 40 phòng, lợi nhuận ổn định...",Bán căn hộ dịch vụ mặt tiền 164 - 166 Nguyễn C...,Nhà,Hồ Chí Minh,Tân Phú,Tân Sơn Nhì,Nguyễn Cửu Đàm,Không rõ,3.500000e+10,130.0,3.0,8.0,15.0,6.0,3.0,2.0,Đông Bắc,Đông Bắc,2025-06-26T11:24:18
1,"Nguyễn Ngọc Lộc Q10, 5tầng BTCT hẻm 5m thông, ...",NGUYỄN NGỌC LỘC - HẺM XE HƠI 5M THÔNG - NHÀ 5 ...,Nhà,Hồ Chí Minh,10,14,Nguyễn Ngọc Lộc,Không rõ,6.500000e+09,24.0,5.0,3.7,15.0,6.0,3.0,4.0,Không rõ,Không rõ,2025-11-17T15:36:52.649000
2,"Nhà hẻm xe hơi ngay Aeon Mall Tân Phú, diện tí...",Nhà siêu hot ngay Aeon Mall Tân Phú đây! Diện ...,Nhà,Hồ Chí Minh,11,Không rõ,Không rõ,Không rõ,6.000000e+09,124.0,3.0,4.0,31.0,6.0,3.0,2.0,Không rõ,Không rõ,2025-09-16T09:56:07.702000
3,"Bán chung cư 4PN, 4WC tại Tôn Thất Thuyết, 64m...","Căn hộ chung cư nằm tại Đường Tôn Thất Thuyết,...",Căn hộ chung cư,Hồ Chí Minh,4,1,Tôn Thất Thuyết,Không rõ,7.500000e+09,64.0,3.0,4.9,15.0,6.0,4.0,4.0,Không rõ,Không rõ,2026-03-20T21:24:50.219000
4,"Bán gấp CC Tây Thạnh DT 72m2, 2 tỷ 7 TL 2PN 2W...","Bán chung cư Tây Thạnh, P. Tây Thạnh, Q. Tân P...",Căn hộ chung cư,Hồ Chí Minh,Tân Phú,Tây Thạnh,C8,Không rõ,2.700000e+09,72.0,3.0,4.9,15.0,6.0,2.0,2.0,Không rõ,Không rõ,2025-10-17T09:40:43.092000
